# Analisis de resultados heuristicos

Este notebook consume los datos generados por `generador_de_datos.ipynb` y organiza la parte analitica de la entrega.

Objetivos de este notebook:
- calcular medias, desviaciones y mejores/peores casos
- comparar metodos por funcion y dimension
- analizar costo en evaluaciones, no solo valor final
- producir tablas y figuras listas para el reporte


In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

BASE_DIR = Path.cwd()
DATA_DIR = BASE_DIR / "datos"
FIGURES_DIR = DATA_DIR / "figuras"
MANIFEST_PATH = DATA_DIR / "manifest_heuristicos.json"

if not DATA_DIR.exists():
    raise FileNotFoundError("No se encontro la carpeta datos dentro de heuristicos.")

if not MANIFEST_PATH.exists():
    raise FileNotFoundError("No se encontro manifest_heuristicos.json. Ejecuta primero generador_de_datos.ipynb.")

FIGURES_DIR.mkdir(parents=True, exist_ok=True)

with MANIFEST_PATH.open("r", encoding="utf-8") as file:
    manifest = json.load(file)

manifest

## Carga de datos

In [ ]:
run_csv_files = sorted(path for path in DATA_DIR.glob("*/*.csv") if not path.name.startswith("resumen_"))
summary_csv_files = sorted(DATA_DIR.glob("*/resumen_*.csv"))
run_json_files = sorted(path for path in DATA_DIR.glob("*/*.json") if not path.name.startswith("resumen_"))

if not run_csv_files:
    raise FileNotFoundError("No se encontraron archivos de corridas dentro de datos/<funcion>/. Ejecuta primero el generador.")

runs_df = pd.concat((pd.read_csv(path) for path in run_csv_files), ignore_index=True)

if summary_csv_files:
    summary_df = pd.concat((pd.read_csv(path) for path in summary_csv_files), ignore_index=True)
else:
    summary_df = pd.DataFrame()

print("Archivos de corridas encontrados:", len(run_csv_files))
print("Archivos de resumen encontrados:", len(summary_csv_files))
runs_df.head()

In [ ]:
numeric_columns = [
    "corrida",
    "semilla",
    "dimension",
    "mejor_valor",
    "distancia_al_optimo",
    "iteraciones",
    "evaluaciones",
]

for column in numeric_columns:
    runs_df[column] = pd.to_numeric(runs_df[column], errors="coerce")

runs_df["caso"] = runs_df["nombre_funcion"] + " " + runs_df["dimension"].astype(int).astype(str) + "D"
runs_df["eficiencia_valor_por_eval"] = runs_df["mejor_valor"] / runs_df["evaluaciones"].replace(0, np.nan)

if summary_df.empty:
    summary_df = (
        runs_df.groupby(["funcion", "nombre_funcion", "metodo", "nombre_metodo", "dimension"], dropna=False)
        .agg(
            n_corridas=("corrida", "count"),
            mejor_valor=("mejor_valor", "min"),
            peor_valor=("mejor_valor", "max"),
            promedio_valor=("mejor_valor", "mean"),
            mediana_valor=("mejor_valor", "median"),
            desviacion_valor=("mejor_valor", "std"),
            mejor_distancia_al_optimo=("distancia_al_optimo", "min"),
            promedio_distancia_al_optimo=("distancia_al_optimo", "mean"),
            promedio_evaluaciones=("evaluaciones", "mean"),
            mediana_evaluaciones=("evaluaciones", "median"),
            promedio_iteraciones=("iteraciones", "mean"),
            mediana_iteraciones=("iteraciones", "median"),
        )
        .reset_index()
    )
    summary_df["desviacion_valor"] = summary_df["desviacion_valor"].fillna(0.0)

summary_df["caso"] = summary_df["nombre_funcion"] + " " + summary_df["dimension"].astype(int).astype(str) + "D"
summary_df.sort_values(["nombre_funcion", "dimension", "metodo"]).reset_index(drop=True)

## Tabla base para reporte

In [ ]:
tabla_reporte = (
    summary_df[
        [
            "nombre_funcion",
            "dimension",
            "nombre_metodo",
            "n_corridas",
            "mejor_valor",
            "promedio_valor",
            "desviacion_valor",
            "promedio_distancia_al_optimo",
            "promedio_evaluaciones",
        ]
    ]
    .sort_values(["nombre_funcion", "dimension", "nombre_metodo"])
    .reset_index(drop=True)
)

tabla_reporte

## Mejores y peores casos

In [ ]:
mejores_casos = runs_df.nsmallest(15, "mejor_valor")[
    ["caso", "nombre_metodo", "corrida", "mejor_valor", "distancia_al_optimo", "evaluaciones"]
]

peores_casos = runs_df.nlargest(15, "mejor_valor")[
    ["caso", "nombre_metodo", "corrida", "mejor_valor", "distancia_al_optimo", "evaluaciones"]
]

print("Mejores casos globales")
display(mejores_casos)
print("Peores casos globales")
display(peores_casos)

## Ranking por caso

Se prioriza valor promedio, luego cercania promedio al optimo y luego costo promedio en evaluaciones.

In [ ]:
ranking_por_caso = (
    summary_df.sort_values(
        ["caso", "promedio_valor", "promedio_distancia_al_optimo", "promedio_evaluaciones"]
    )
    .groupby("caso", as_index=False)
    .first()[
        [
            "caso",
            "nombre_metodo",
            "promedio_valor",
            "promedio_distancia_al_optimo",
            "promedio_evaluaciones",
        ]
    ]
)

ranking_por_caso

## Figuras para reporte

In [ ]:
method_colors = {
    "Algoritmo evolutivo": "#4C78A8",
    "PSO": "#F58518",
    "Evolucion diferencial": "#54A24B",
}

def save_current_figure(filename: str) -> Path:
    path = FIGURES_DIR / filename
    plt.savefig(path, dpi=300, bbox_inches="tight")
    return path


### 1. Barras comparativas por metodo

In [ ]:
promedio_por_metodo = (
    summary_df.groupby("nombre_metodo", as_index=False)
    .agg(
        promedio_valor=("promedio_valor", "mean"),
        promedio_evaluaciones=("promedio_evaluaciones", "mean"),
        promedio_distancia=("promedio_distancia_al_optimo", "mean"),
    )
)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for ax, metric, title in [
    (axes[0], "promedio_valor", "Valor final promedio"),
    (axes[1], "promedio_evaluaciones", "Evaluaciones promedio"),
    (axes[2], "promedio_distancia", "Distancia promedio al optimo"),
]:
    colors = [method_colors.get(name, "#777777") for name in promedio_por_metodo["nombre_metodo"]]
    ax.bar(promedio_por_metodo["nombre_metodo"], promedio_por_metodo[metric], color=colors)
    ax.set_title(title)
    ax.tick_params(axis="x", rotation=20)
    ax.grid(axis="y", alpha=0.3)

plt.tight_layout()
fig_path = save_current_figure("01_barras_resumen_por_metodo.png")
plt.show()
fig_path

### 2. Heatmap comparativo por caso y metodo

In [ ]:
pivot_valor = summary_df.pivot(index="caso", columns="nombre_metodo", values="promedio_valor").sort_index()
pivot_eval = summary_df.pivot(index="caso", columns="nombre_metodo", values="promedio_evaluaciones").sort_index()

fig, axes = plt.subplots(1, 2, figsize=(16, 7))

im1 = axes[0].imshow(pivot_valor.values, aspect="auto", cmap="viridis")
axes[0].set_title("Promedio del valor final")
axes[0].set_xticks(range(len(pivot_valor.columns)))
axes[0].set_xticklabels(pivot_valor.columns, rotation=20)
axes[0].set_yticks(range(len(pivot_valor.index)))
axes[0].set_yticklabels(pivot_valor.index)
fig.colorbar(im1, ax=axes[0], fraction=0.046, pad=0.04)

im2 = axes[1].imshow(pivot_eval.values, aspect="auto", cmap="magma")
axes[1].set_title("Promedio de evaluaciones")
axes[1].set_xticks(range(len(pivot_eval.columns)))
axes[1].set_xticklabels(pivot_eval.columns, rotation=20)
axes[1].set_yticks(range(len(pivot_eval.index)))
axes[1].set_yticklabels(pivot_eval.index)
fig.colorbar(im2, ax=axes[1], fraction=0.046, pad=0.04)

plt.tight_layout()
fig_path = save_current_figure("02_heatmap_casos_vs_metodos.png")
plt.show()
fig_path

### 3. Distribuciones por metodo

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5))
method_names = list(dict.fromkeys(runs_df["nombre_metodo"]))

for method_name in method_names:
    subset = runs_df[runs_df["nombre_metodo"] == method_name]
    color = method_colors.get(method_name, "#777777")
    axes[0].hist(subset["mejor_valor"], bins=20, alpha=0.5, label=method_name, color=color)
    axes[1].hist(subset["evaluaciones"], bins=20, alpha=0.5, label=method_name, color=color)

axes[0].set_title("Distribucion del mejor valor")
axes[0].set_xlabel("Mejor valor")
axes[0].set_ylabel("Frecuencia")
axes[0].grid(alpha=0.3)

axes[1].set_title("Distribucion de evaluaciones")
axes[1].set_xlabel("Evaluaciones")
axes[1].set_ylabel("Frecuencia")
axes[1].grid(alpha=0.3)

axes[0].legend()
axes[1].legend()
plt.tight_layout()
fig_path = save_current_figure("03_histogramas_por_metodo.png")
plt.show()
fig_path

### 4. Comparacion por caso

In [ ]:
ordered_cases = sorted(summary_df["caso"].unique())
n_cases = len(ordered_cases)
fig, axes = plt.subplots(n_cases, 2, figsize=(14, max(4 * n_cases, 10)))

if n_cases == 1:
    axes = np.array([axes])

for row_index, case_name in enumerate(ordered_cases):
    case_data = summary_df[summary_df["caso"] == case_name].sort_values("nombre_metodo")
    colors = [method_colors.get(name, "#777777") for name in case_data["nombre_metodo"]]

    axes[row_index, 0].bar(case_data["nombre_metodo"], case_data["promedio_valor"], color=colors)
    axes[row_index, 0].set_title(f"{case_name} - Valor promedio")
    axes[row_index, 0].tick_params(axis="x", rotation=20)
    axes[row_index, 0].grid(axis="y", alpha=0.3)

    axes[row_index, 1].bar(case_data["nombre_metodo"], case_data["promedio_evaluaciones"], color=colors)
    axes[row_index, 1].set_title(f"{case_name} - Evaluaciones promedio")
    axes[row_index, 1].tick_params(axis="x", rotation=20)
    axes[row_index, 1].grid(axis="y", alpha=0.3)

plt.tight_layout()
fig_path = save_current_figure("04_comparacion_por_caso.png")
plt.show()
fig_path

### 5. Graficas de convergencia

Estas curvas requieren que cada archivo JSON de corridas incluya `best_values_history`. Si todavia no aparece ese campo, hay que volver a ejecutar el generador.

In [ ]:
def load_json_runs() -> list[dict]:
    all_rows = []
    for json_path in run_json_files:
        with json_path.open("r", encoding="utf-8") as file:
            rows = json.load(file)
        for row in rows:
            row["source_file"] = json_path.name
            all_rows.append(row)
    return all_rows

json_runs = load_json_runs()
has_histories = any("best_values_history" in row for row in json_runs)
print("Hay historiales de convergencia disponibles:", has_histories)

In [ ]:
if has_histories:
    convergence_rows = [row for row in json_runs if "best_values_history" in row and row["best_values_history"]]
    convergence_cases = sorted({f"{row['nombre_funcion']} {row['dimension']}D" for row in convergence_rows})

    if convergence_cases:
        selected_cases = convergence_cases[: min(6, len(convergence_cases))]
        n_cols = 2
        n_rows = int(np.ceil(len(selected_cases) / n_cols))
        fig, axes = plt.subplots(n_rows, n_cols, figsize=(14, 5 * n_rows))
        axes = np.array(axes).reshape(-1)

        for ax, case_name in zip(axes, selected_cases):
            case_rows = [
                row for row in convergence_rows
                if f"{row['nombre_funcion']} {row['dimension']}D" == case_name
            ]

            for method_name in sorted({row['nombre_metodo'] for row in case_rows}):
                method_histories = [
                    np.asarray(row["best_values_history"], dtype=float)
                    for row in case_rows
                    if row["nombre_metodo"] == method_name
                ]

                min_length = min(len(history) for history in method_histories)
                aligned = np.vstack([history[:min_length] for history in method_histories])
                mean_history = aligned.mean(axis=0)
                color = method_colors.get(method_name, "#777777")
                ax.plot(mean_history, label=method_name, linewidth=2, color=color)

            ax.set_title(f"Convergencia - {case_name}")
            ax.set_xlabel("Iteracion")
            ax.set_ylabel("Mejor valor medio")
            ax.set_yscale("symlog")
            ax.grid(alpha=0.3)
            ax.legend()

        for ax in axes[len(selected_cases):]:
            ax.axis("off")

        plt.tight_layout()
        fig_path = save_current_figure("05_convergencia_por_caso.png")
        plt.show()
        display(fig_path)
    else:
        print("No hay casos con historiales de convergencia para graficar.")
else:
    print("No se encontraron historiales. Vuelve a ejecutar generador_de_datos.ipynb despues de guardar best_values_history.")

## Exportables utiles para el reporte

In [ ]:
tabla_reporte_path = FIGURES_DIR / "tabla_resumen_heuristicos.csv"
ranking_path = FIGURES_DIR / "ranking_por_caso.csv"
mejores_path = FIGURES_DIR / "mejores_casos.csv"
peores_path = FIGURES_DIR / "peores_casos.csv"

tabla_reporte.to_csv(tabla_reporte_path, index=False, encoding="utf-8")
ranking_por_caso.to_csv(ranking_path, index=False, encoding="utf-8")
mejores_casos.to_csv(mejores_path, index=False, encoding="utf-8")
peores_casos.to_csv(peores_path, index=False, encoding="utf-8")

print("Archivos exportados:")
print("-", tabla_reporte_path)
print("-", ranking_path)
print("-", mejores_path)
print("-", peores_path)